
# Bybit Kline Data Fetching and Updating in Jupyter Notebook

This notebook is designed to fetch and update data from the Bybit API, save it to parquet files, and maintain rate limits. The notebook is divided into several sections:
1. Setting up libraries and API keys.
2. Get List of  Available USDT Tickres
3. Defining functions for data formatting and fetching.
4. Creating new data files.
5. Updating existing data files.


## 1: Libraries, API Keys and folder 

In [8]:

# Importing necessary libraries
from pybit.unified_trading import HTTP
import pandas as pd
import datetime as dt
import time
import os

# Setting up the API key and secret
key = '6yXHlbODQ0TaGnp8cA'
secret = 'HIRlZdCIkyDrML3loAvSfNcBrJO2mG9xS7ts'

# Folder for saving data or update data
folder = 'Bybit_Kline_Data'


#type of data
category = 'linear'  # linear = Perpetuals, spot = spot


# Initializing the session with your API key and secret
session = HTTP(api_key=key, api_secret=secret, testnet=False)



## 2:  Get List of  Available USDT Tickres 

In [9]:

result = session.get_tickers(
    category=category).get('result')['list']

symbols = [asset['symbol'] for asset in result if asset['symbol'].endswith('USDT')]
print(len(symbols), 'symbols found')

print(symbols)

425 symbols found
['10000000AIDOGEUSDT', '1000000BABYDOGEUSDT', '1000000MOGUSDT', '1000000PEIPEIUSDT', '10000COQUSDT', '10000LADYSUSDT', '10000SATSUSDT', '10000WENUSDT', '10000WHYUSDT', '1000APUUSDT', '1000BONKUSDT', '1000BTTUSDT', '1000CATSUSDT', '1000CATUSDT', '1000FLOKIUSDT', '1000LUNCUSDT', '1000MUMUUSDT', '1000NEIROCTOUSDT', '1000PEPEUSDT', '1000RATSUSDT', '1000TURBOUSDT', '1000XECUSDT', '1000XUSDT', '1CATUSDT', '1INCHUSDT', 'A8USDT', 'AAVEUSDT', 'ACEUSDT', 'ACHUSDT', 'ADAUSDT', 'AERGOUSDT', 'AEROUSDT', 'AEVOUSDT', 'AGIUSDT', 'AGLDUSDT', 'AIOZUSDT', 'AIUSDT', 'AKROUSDT', 'AKTUSDT', 'ALEOUSDT', 'ALGOUSDT', 'ALICEUSDT', 'ALPACAUSDT', 'ALPHAUSDT', 'ALTUSDT', 'AMBUSDT', 'ANKRUSDT', 'APEUSDT', 'API3USDT', 'APTUSDT', 'ARBUSDT', 'ARKMUSDT', 'ARKUSDT', 'ARPAUSDT', 'ARUSDT', 'ASTRUSDT', 'ATAUSDT', 'ATHUSDT', 'ATOMUSDT', 'AUCTIONUSDT', 'AUDIOUSDT', 'AVAILUSDT', 'AVAXUSDT', 'AXLUSDT', 'AXSUSDT', 'BADGERUSDT', 'BAKEUSDT', 'BALUSDT', 'BANANAUSDT', 'BANDUSDT', 'BATUSDT', 'BBUSDT', 'BCHUSDT', 'B

## 3: Defining Functions

In [10]:

# Function to format the response data
def format_data(response):
    data = response.get('list', None)
    if not data:
        return

    data = pd.DataFrame(data,
                        columns=[
                            'timestamp',
                            'open',
                            'high',
                            'low',
                            'close',
                            'volume',
                            'turnover'
                        ],
                       )
    f = lambda x: dt.datetime.utcfromtimestamp(int(x) / 1000)
    data.index = data.timestamp.apply(f)
    return data[::-1].apply(pd.to_numeric)

# Function to get the last timestamp from the dataframe
def get_last_timestamp(df):
    return int(df.timestamp[-1:].values[0])

# Function to fetch symbol data with rate limit checks
def get_symbol_data(symbol, start_time, interval, rate_limits):
    start = int(start_time.timestamp() * 1000)
    df = pd.DataFrame()

    while True:
        # Fetch the data
        response = session.get_kline(category=category,
                                     symbol=symbol,
                                     start=start,
                                     interval=interval).get('result')

        latest = format_data(response)

        if not isinstance(latest, pd.DataFrame):
            break

        start = get_last_timestamp(latest)

        # Update the inner rate limit (600 requests per 5-second window)
        time_since_inner_window_start = time.time() - rate_limits['inner_window_start']
        if rate_limits['inner_requests'] >= 600 or (time_since_inner_window_start >= 5 and rate_limits['inner_requests'] > 0):
            time.sleep(max(0, 5 - time_since_inner_window_start))
            rate_limits['inner_window_start'] = time.time()
            rate_limits['inner_requests'] = 0

        # Add the fetched data to the dataframe and sleep for a short duration
        df = pd.concat([df, latest])
        rate_limits['inner_requests'] += 1
        time.sleep(0.1)

        if len(latest) == 1:
            break

    # Ensure the directory exists
    ensure_directory_exists(folder)
    
    # Save the data to a parquet file
    df.drop_duplicates(subset=['timestamp'], keep='last', inplace=True)
    df.to_parquet(f'{folder}/{symbol}.parquet')
    print(f'{symbol} saved to parquet')
#     print(df)

# Function to get the symbol names from a folder
def get_symbol_names(folder):
    return [os.path.splitext(file_name)[0] for file_name in os.listdir(folder) if file_name.endswith('.parquet')]

# Function to update symbol data
def update_symbol_data(symbol, interval, rate_limits):
    # Read existing data
    try:
        df = pd.read_parquet(f'{folder}/{symbol}.parquet')
    except FileNotFoundError:
        # If the file doesn't exist, start from the specified start_time
        start_time = dt.datetime(2020, 1, 1)
    else:
        # If the file exists, start from the last timestamp in the file
        start_time = dt.datetime.fromtimestamp(df.timestamp[-1:].values[0] / 1000 + 1)

    # Fetch new data
    new_data = get_symbol_data(symbol, start_time, interval, rate_limits)

    if isinstance(new_data, pd.DataFrame):
        # Concatenate old and new data
        df = pd.concat([df, new_data])

        # Drop duplicates and save the data
        df.drop_duplicates(subset=['timestamp'], keep='last', inplace=True)
        ensure_directory_exists(folder)
        df.to_parquet(f'{folder}/{symbol}.parquet')
        print(f'{symbol} saved to parquet')

# Function to set up rate limit tracking variables
def setup_rate_limits():
    return {
        'outer_window_start': time.time(),
        'outer_requests': 0,
        'inner_window_start': time.time(),
        'inner_requests': 0
    }


# Function to ensure the directory exists
def ensure_directory_exists(folder):
    if not os.path.exists(folder):
        os.makedirs(folder)


## 5: Get data and creating Files

### Select parameters 

In [11]:
# Starting time for fetching data
start_time = dt.datetime(2024, 10, 27)

# Interval for fetching data

interval = '60' # Kline interval. Options: 1,3,5,15,30,60,120,240,360,720,D,M,W


# List of symbols to fetch data for
# symbols = ['BTCUSDT', 'ETHBTC']  # Uncomment this line and add the desired ticker or leave it commented to import data for all tickers.

### get data

In [12]:

# Ensure the directory exists
ensure_directory_exists(folder)

# Set up rate limit tracking variables
rate_limits = setup_rate_limits()



# Fetch data for each symbol with rate limit checks
for symbol in symbols:
    print(f'Collecting data for {symbol}')
    get_symbol_data(symbol, start_time, interval, rate_limits)

    # Update the outer rate limit (10 requests per second)
    rate_limits['outer_requests'] += 1
    time_since_outer_window_start = time.time() - rate_limits['outer_window_start']
    if rate_limits['outer_requests'] >= 10 or (time_since_outer_window_start >= 1 and rate_limits['outer_requests'] > 0):
        time.sleep(max(0, 1 - time_since_outer_window_start))
        rate_limits['outer_window_start'] = time.time()


10000000AIDOGEUSDT saved to parquet
1000000BABYDOGEUSDT saved to parquet
1000000MOGUSDT saved to parquet
1000000PEIPEIUSDT saved to parquet
10000COQUSDT saved to parquet
10000LADYSUSDT saved to parquet
10000SATSUSDT saved to parquet
10000WENUSDT saved to parquet
10000WHYUSDT saved to parquet
1000APUUSDT saved to parquet
1000BONKUSDT saved to parquet
1000BTTUSDT saved to parquet
1000CATSUSDT saved to parquet
1000CATUSDT saved to parquet
1000FLOKIUSDT saved to parquet
1000LUNCUSDT saved to parquet
1000MUMUUSDT saved to parquet
1000NEIROCTOUSDT saved to parquet
1000PEPEUSDT saved to parquet
1000RATSUSDT saved to parquet
1000TURBOUSDT saved to parquet
1000XECUSDT saved to parquet
1000XUSDT saved to parquet
1CATUSDT saved to parquet
1INCHUSDT saved to parquet
A8USDT saved to parquet
AAVEUSDT saved to parquet
ACEUSDT saved to parquet
ACHUSDT saved to parquet
ADAUSDT saved to parquet
AERGOUSDT saved to parquet
AEROUSDT saved to parquet
AEVOUSDT saved to parquet
AGIUSDT saved to parquet
AGLDUS

KeyboardInterrupt: 

## 6: Updating Files

### Select parameters 

In [13]:

# Interval for fetching data
interval = '1' # Kline interval. Options: 1,3,5,15,30,60,120,240,360,720,D,M,W

### Updat data

In [14]:

# Ensure the directory exists
ensure_directory_exists(folder)

# Set up rate limit tracking variables
rate_limits = setup_rate_limits()

# Get the symbol names from the bybit_data folder
symbol_names = get_symbol_names(folder)

# Update data for each symbol with rate limit checks
for symbol in symbol_names:
    print(f'Collecting data for {symbol}')
    update_symbol_data(symbol, interval, rate_limits)

    # Update the outer rate limit
    rate_limits['outer_requests'] += 1
    time_since_outer_window_start = time.time() - rate_limits['outer_window_start']
    if rate_limits['outer_requests'] >= 10 or (time_since_outer_window_start >= 1 and rate_limits['outer_requests'] > 0):
        time.sleep(max(0, 1 - time_since_outer_window_start))
        rate_limits['outer_window_start'] = time.time()

print('done')


10000000AIDOGEUSDT saved to parquet
1000000BABYDOGEUSDT saved to parquet
1000000MOGUSDT saved to parquet
1000000PEIPEIUSDT saved to parquet


KeyboardInterrupt: 